**Code for assigning anatomical labels to ROIs from chosen brain atlas.**

- Reads current atlas parameters from config.yaml file.

- Uses Harvard-Oxford brain atlas for ANATOMICAL labels.

- Uses YEO-7 / YEO-17 brain atlases for FUNCTIONAL labels ("coarse" and "fine", respectively).

- Computes ROI center-of-mass coordinates to assign hemisphere labels.

- Includes manual input cell for user-editing of labels (fully optional).

--------

In [ ]:
### LOAD CONFIG.YAML:

import yaml, os
from pathlib import Path

CONFIG_PATH = Path.cwd() / "config.yaml"

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH.resolve()}")

with open(CONFIG_PATH, 'r') as config_file:
    config = yaml.safe_load(config_file)

# config

In [ ]:
import os
import csv
import nibabel as nib
import numpy as np
import pandas as pd
from nilearn import datasets, image, plotting
from scipy.ndimage import center_of_mass
from nilearn.image import coord_transform
import matplotlib.pyplot as plt
import textwrap

In [ ]:
# __________________________________________________________________________________________________________
### LOAD PARAMETERS:

# General parameters:
HARD_STOP    = config['hard_errors']
RANDOM_SEED  = config['random_seed']


# Parameters specifying target parcellation atlas:
ATLAS = config['parcellation']['atlas']
NUM_ROIS = config['parcellation']['n_rois']

if ATLAS.strip().lower() == 'craddock':
    CRADDOCK_PATH = config['atlases']['Craddock']['craddock_dir']
    CRADDOCK_MNI  = config['atlases']['Craddock']['canonical_mni']
    CRADDOCK_TARGET_FILE = CRADDOCK_PATH + f"Craddock-{NUM_ROIS}_ROIs.nii"

elif ATLAS.strip().lower() == 'schaefer':
    NETWORK_SCALE = config['parcellation']['network_scale']
    RESOLUTION = int(config.get("atlases", {}).get("Schaefer", {}).get("resolution", 2))


LABEL_OUTPUT_DIR = Path(config['parcellation']['label_output_dir'])

EXPORT_FULL_ROI_REPORTS = config['parcellation']['save_ROI_detail_plots']


# __________________________________________________________________________________________________________
### SET FILEPATHS:

BASE_DIRECTORY = Path(config['root_output_directory'])

RUN_MANIFEST_PATH    = BASE_DIRECTORY / 'subject_manifest.csv'
fMRI_PARAMETERS_PATH = BASE_DIRECTORY / 'fMRI_manifest.csv'


### OUTPUTS:

ROOT_OUTPUT_DIR = Path(BASE_DIRECTORY)

LABEL_OUTPUT_PATH = ROOT_OUTPUT_DIR / LABEL_OUTPUT_DIR
LABEL_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

Load original parcellation map (i.e. Craddock / Schaefer / MIST):

In [ ]:
# __________________________________________________________________________________________________________
### LOAD PARCELLATION ATLAS LABEL IMAGE (MNI SPACE; SUBJECT-AGNOSTIC)

atlas_family = str(ATLAS).strip()
atlas_family_lower = atlas_family.lower()

if not isinstance(NUM_ROIS, int) or NUM_ROIS <= 0:
    raise ValueError(f"parcellation.n_rois must be a positive integer. Got: {NUM_ROIS}")

parcellation_label_image = None
parcellation_roi_names = None  # optional; populated if available (e.g., Schaefer)
atlas_tag = None

# -------------------------
# CRADDOCK (local file)
# -------------------------
if atlas_family_lower == "craddock":
    craddock_directory = str(config["atlases"]["Craddock"]["craddock_dir"])
    if not craddock_directory.endswith(os.sep):
        craddock_directory = craddock_directory + os.sep

    craddock_target_file = craddock_directory + f"Craddock-{NUM_ROIS}_ROIs.nii"
    craddock_target_path = Path(craddock_target_file).expanduser()

    if not craddock_target_path.exists():
        raise FileNotFoundError(
            f"Craddock atlas file not found: {craddock_target_path}\n"
            "Confirm the filename convention and directory are correct.")

    parcellation_label_image = nib.load(str(craddock_target_path))
    atlas_tag = f"Craddock-{NUM_ROIS:03d}"

# -------------------------
# SCHAEFER (nilearn fetch)
# -------------------------
elif atlas_family_lower == "schaefer":
    network_scale_value = config["parcellation"].get("network_scale", None)
    if network_scale_value is None:
        raise ValueError("Schaefer selected but parcellation.network_scale is missing (expected an integer).")

    # Allowed scales validation via YAML (atlases.Schaefer.allowed_scales)
    allowed_scales_mapping = config.get("atlases", {}).get("Schaefer", {}).get("allowed_scales", {})
    allowed_scales_for_n_rois = allowed_scales_mapping.get(int(NUM_ROIS), None)

    if allowed_scales_for_n_rois is None:
        message = (
            "Schaefer atlas selected, but atlases.Schaefer.allowed_scales has no entry for "
            f"n_rois={int(NUM_ROIS)}.\n"
            f"Available n_rois keys: {sorted(list(allowed_scales_mapping.keys()))}")
        if HARD_STOP:
            raise KeyError(message)
        else:
            print("[WARN] " + message)

    if allowed_scales_for_n_rois is not None and int(network_scale_value) not in list(allowed_scales_for_n_rois):
        raise ValueError(
            f"Invalid Schaefer network_scale={int(network_scale_value)} for n_rois={int(NUM_ROIS)}.\n"
            f"Allowed values per YAML atlases.Schaefer.allowed_scales[{int(NUM_ROIS)}]: {allowed_scales_for_n_rois}")

    schaefer = datasets.fetch_atlas_schaefer_2018(
        n_rois=int(NUM_ROIS),
        yeo_networks=int(network_scale_value),
        resolution_mm=int(RESOLUTION),)

    parcellation_label_image = nib.load(schaefer["maps"])
    parcellation_roi_names = schaefer.get("labels", None)
    atlas_tag = f"Schaefer-{NUM_ROIS:03d}_Yeo{int(network_scale_value):02d}_res{int(RESOLUTION)}mm"

# -------------------------
# MIST (multiscale; nilearn BASC dataset)
# -------------------------
elif atlas_family_lower == "mist":
    # Note: MIST/BASC multiscale availability is determined by the dataset source.
    # If you use scales not provided by nilearn (e.g., 588/799/1228), you will need to
    # supply canonical MNI files via YAML similarly to Craddock.
    basc = datasets.fetch_atlas_basc_multiscale_2015()
    basc_key = f"scale{int(NUM_ROIS):03d}"

    if basc_key not in basc:
        raise KeyError(
            f"MIST selected with n_rois={int(NUM_ROIS)}, but nilearn BASC multiscale does not provide '{basc_key}'.\n"
            "If your pipeline uses a custom MIST distribution for this scale, provide a canonical MNI NIfTI file path "
            "in YAML (e.g., atlases.MIST.paths[<n_rois>]) and load it from disk.")

    parcellation_label_image = nib.load(basc[basc_key])
    atlas_tag = f"MIST-{NUM_ROIS:03d}"

else:
    raise ValueError(f"Unsupported parcellation atlas: '{atlas_family}'. Expected Craddock, Schaefer, or MIST.")

# -------------------------
# Robust integer label conversion
# -------------------------
parcellation_label_float = parcellation_label_image.get_fdata()
parcellation_label_rounded = np.rint(parcellation_label_float)
parcellation_label_rounded[~np.isfinite(parcellation_label_rounded)] = 0
parcellation_label_data_int = parcellation_label_rounded.astype(np.int32)

unique_labels = np.unique(parcellation_label_data_int)
roi_labels_present = sorted([int(label_value) for label_value in unique_labels if int(label_value) != 0])

if len(roi_labels_present) == 0:
    raise RuntimeError("Parcellation label image contains no non-zero ROI labels.")

# Enforce expected ROI count (warn or hard-stop based on HARD_STOP)
if len(roi_labels_present) != int(NUM_ROIS):
    message = (
        f"ROI count mismatch for atlas_tag={atlas_tag}: expected {int(NUM_ROIS)} non-zero labels, "
        f"found {len(roi_labels_present)}.\n"
        f"Min label: {min(roi_labels_present)}, Max label: {max(roi_labels_present)}\n"
        "This can happen if the atlas file/scale is not the expected one, or if labels are missing/non-contiguous.")
    if HARD_STOP:
        raise RuntimeError(message)
    else:
        print("[WARN] " + message)

# Convenience values for later steps:
parcellation_affine = parcellation_label_image.affine
roi_id_string_width = len(str(max(roi_labels_present)))

print(f"[INFO] Loaded parcellation atlas: {atlas_tag}")
print(f"[INFO] Label image shape: {parcellation_label_data_int.shape}")
print(f"[INFO] ROI labels present (n={len(roi_labels_present)}): {roi_labels_present[:10]}{'...' if len(roi_labels_present) > 10 else ''}")
print(f"[INFO] Suggested roi_id string width for optional zero-padding: {roi_id_string_width}")

Next, load Harvard-Oxford and YEO atlases, and resample to common frame of reference, with additional visual QC checks (overlays on canonical MNI reference anatomical brain):

In [ ]:
# __________________________________________________________________________________________________________
### LOAD HARVARD–OXFORD + YEO ATLASES AND RESAMPLE TO PARCELLATION GRID (MNI)

# Background template for QC (MNI anatomical)
mni_anatomical_template = datasets.load_mni152_template()

# -------------------------
# Quick affine / voxel-size comparisons (before resampling)
# -------------------------
def _voxel_sizes_from_affine(affine_matrix):
    affine_matrix = np.asarray(affine_matrix)
    return np.sqrt(np.sum(affine_matrix[:3, :3] ** 2, axis=0))

parcellation_voxel_sizes_mm = _voxel_sizes_from_affine(parcellation_label_image.affine)
template_voxel_sizes_mm = _voxel_sizes_from_affine(mni_anatomical_template.affine)

print("[INFO] Affine / voxel-size summary (mm):")
print(f"       Parcellation label image shape: {parcellation_label_image.shape}")
print(f"       Parcellation voxel sizes (mm):  {parcellation_voxel_sizes_mm}")
print(f"       MNI template shape:             {mni_anatomical_template.shape}")
print(f"       MNI template voxel sizes (mm):  {template_voxel_sizes_mm}")
print(f"       Parcellation affine:\n{parcellation_label_image.affine}")
print(f"       MNI template affine:\n{mni_anatomical_template.affine}")

# -------------------------
# Harvard–Oxford (max-prob integer atlases)
# -------------------------
harvard_oxford_cortical_maxprob = datasets.fetch_atlas_harvard_oxford("cort-maxprob-thr25-2mm")
harvard_oxford_subcortical_maxprob = datasets.fetch_atlas_harvard_oxford("sub-maxprob-thr25-2mm")

print("\n[INFO] Harvard-Oxford source atlas maps (pre-resampling):")
print(f"       HO cortical map type: {type(harvard_oxford_cortical_maxprob.maps)}")
print(f"       HO subcortical map type: {type(harvard_oxford_subcortical_maxprob.maps)}")

# Resample to the parcellation grid (nearest to preserve integer labels)
harvard_oxford_cortical_maxprob_resampled_image = image.resample_to_img(
    source_img=harvard_oxford_cortical_maxprob.maps,
    target_img=parcellation_label_image,
    interpolation="nearest")

harvard_oxford_subcortical_maxprob_resampled_image = image.resample_to_img(
    source_img=harvard_oxford_subcortical_maxprob.maps,
    target_img=parcellation_label_image,
    interpolation="nearest")

harvard_oxford_cortical_maxprob_data_int = np.rint(
    harvard_oxford_cortical_maxprob_resampled_image.get_fdata()).astype(np.int32)

harvard_oxford_subcortical_maxprob_data_int = np.rint(
    harvard_oxford_subcortical_maxprob_resampled_image.get_fdata()).astype(np.int32)

harvard_oxford_cortical_region_names = list(harvard_oxford_cortical_maxprob.labels)
harvard_oxford_subcortical_region_names = list(harvard_oxford_subcortical_maxprob.labels)

# Voxel sizes after resampling (should match parcellation grid)
ho_cortical_voxel_sizes_mm = _voxel_sizes_from_affine(harvard_oxford_cortical_maxprob_resampled_image.affine)
ho_subcortical_voxel_sizes_mm = _voxel_sizes_from_affine(harvard_oxford_subcortical_maxprob_resampled_image.affine)

print("\n[INFO] Harvard-Oxford max-prob atlases loaded and resampled to parcellation grid:")
print(f"       HO cortical resampled shape:     {harvard_oxford_cortical_maxprob_data_int.shape}")
print(f"       HO cortical voxel sizes (mm):    {ho_cortical_voxel_sizes_mm}")
print(f"       HO subcortical resampled shape:  {harvard_oxford_subcortical_maxprob_data_int.shape}")
print(f"       HO subcortical voxel sizes (mm): {ho_subcortical_voxel_sizes_mm}")
print(f"       HO cortical resampled affine:\n{harvard_oxford_cortical_maxprob_resampled_image.affine}")
print(f"       HO subcortical resampled affine:\n{harvard_oxford_subcortical_maxprob_resampled_image.affine}")

# -------------------------
# Yeo 2011 networks (integer-labeled maps)
# -------------------------
yeo_2011 = datasets.fetch_atlas_yeo_2011()
yeo_7_networks_image = yeo_2011.thick_7
yeo_17_networks_image = yeo_2011.thick_17

yeo_7_networks_resampled_image = image.resample_to_img(
    source_img=yeo_7_networks_image,
    target_img=parcellation_label_image,
    interpolation="nearest")

yeo_17_networks_resampled_image = image.resample_to_img(
    source_img=yeo_17_networks_image,
    target_img=parcellation_label_image,
    interpolation="nearest")

# NOTE: Some nilearn versions yield Yeo maps as 4D images, with a trailing singleton dimension (X, Y, Z, 1).
# The downstream labeling code expects 3D integer label volumes, so we explicitly coerce to 3D here:
if len(yeo_7_networks_resampled_image.shape) == 4 and int(yeo_7_networks_resampled_image.shape[3]) == 1:
    yeo_7_networks_resampled_image = image.index_img(yeo_7_networks_resampled_image, 0)
if len(yeo_17_networks_resampled_image.shape) == 4 and int(yeo_17_networks_resampled_image.shape[3]) == 1:
    yeo_17_networks_resampled_image = image.index_img(yeo_17_networks_resampled_image, 0)

yeo_7_networks_data_int = np.rint(yeo_7_networks_resampled_image.get_fdata()).astype(np.int32)
yeo_17_networks_data_int = np.rint(yeo_17_networks_resampled_image.get_fdata()).astype(np.int32)

yeo7_voxel_sizes_mm = _voxel_sizes_from_affine(yeo_7_networks_resampled_image.affine)
yeo17_voxel_sizes_mm = _voxel_sizes_from_affine(yeo_17_networks_resampled_image.affine)

print("\n[INFO] Yeo 2011 7- and 17-network atlases loaded and resampled to parcellation grid:")
print(f"       Yeo-7 resampled shape:           {yeo_7_networks_data_int.shape}")
print(f"       Yeo-7 voxel sizes (mm):          {yeo7_voxel_sizes_mm}")
print(f"       Yeo-17 resampled shape:          {yeo_17_networks_data_int.shape}")
print(f"       Yeo-17 voxel sizes (mm):         {yeo17_voxel_sizes_mm}")
print(f"       Yeo-7 resampled affine:\n{yeo_7_networks_resampled_image.affine}")
print(f"       Yeo-17 resampled affine:\n{yeo_17_networks_resampled_image.affine}")

# -------------------------
# Quick visual QC overlays
# -------------------------
# # QC overlay 1: parcellation ROI overlay on MNI anatomy
# print("\n[QC] Displaying parcellation ROI overlay on MNI anatomical template...")
# plotting.plot_roi(
#     roi_img=parcellation_label_image,
#     bg_img=mni_anatomical_template,
#     display_mode="ortho",
#     cut_coords=(0, 0, 0),
#     title=f"QC: Parcellation overlay ({atlas_tag}) on MNI template")

# QC overlay 2: Harvard–Oxford cortical max-prob overlay (resampled)
print("[QC] Displaying Harvard-Oxford cortical max-prob overlay (resampled) on MNI template...")
plotting.plot_roi(
    roi_img=harvard_oxford_cortical_maxprob_resampled_image,
    bg_img=mni_anatomical_template,
    display_mode="ortho",
    cut_coords=(0, 0, 0),
    title="Resampled Harvard-Oxford cortical maxProb on MNI template:")

# QC overlay 3: Yeo-7 overlay (resampled)
print("[QC] Displaying Yeo-7 overlay (resampled) on MNI template...")
plotting.plot_roi(
    roi_img=yeo_7_networks_resampled_image,
    bg_img=mni_anatomical_template,
    display_mode="ortho",
    cut_coords=(0, 0, 0),
    title="Resampled Yeo-7 on MNI template:")

plt.show()

In [ ]:
# __________________________________________________________________________________________________________
### COMPUTE PER-ROI OVERLAPS AND BUILD LABEL TABLE
#
#   (1) For each ROI mask in the parcellation label map, compute:
#       - centroid (x,y,z) in MNI space + hemisphere
#       - top-3 Harvard–Oxford max-prob labels by voxel overlap (coverage %)
#       - top-3 Yeo-7 labels by voxel overlap (coverage %)
#       - top-3 Yeo-17 labels by voxel overlap (coverage %)
#   (2) Mark as ambiguous if primary HO coverage < threshold
#   (3) Create manual_label column if missing/empty

PRIMARY_LOW_THRESH_PERCENT = 50  # mark ROI as ambiguous if primary anatomy coverage < this

# "Catch-all" / non-specific labels to ignore for anatomy overlap (same spirit as old code)
GENERIC_ANATOMY_LABELS = {
    "Left Cerebral Cortex", "Right Cerebral Cortex", "Cerebral Cortex",
    "Left Cerebral White Matter", "Right Cerebral White Matter", "Cerebral White Matter",
    "Left Lateral Ventricle", "Right Lateral Ventricle"}

# Yeo friendly names (same mapping as old code)
yeo_7_network_names = {
    1: "Visual",
    2: "Somatomotor",
    3: "Dorsal Attention",
    4: "Ventral Attention",
    5: "Limbic System",
    6: "Frontoparietal",
    7: "Default Mode Network (DMN)"}

yeo_17_network_names = {
    1: "Visual A", 2: "Visual B",
    3: "Somatomotor A", 4: "Somatomotor B",
    5: "Dorsal Attention A", 6: "Dorsal Attention B",
    7: "Salience/Ventral Attn A", 8: "Salience/Ventral Attn B",
    9: "Limbic System A", 10: "Limbic System B",
    11: "Executive Control A", 12: "Executive Control B", 13: "Executive Control C",
    14: "Default Mode Network A", 15: "Default Mode Network B",
    16: "Default Mode Network C", 17: "Default Mode Network D"}

def _percent(value_fraction):
    return int(np.rint(100.0 * float(value_fraction)))

def _compute_centroid_xyz_and_hemi(mask_3d, affine_matrix, midline_tolerance_mm=5.0):
    com_ijk = center_of_mass(mask_3d)
    x, y, z = coord_transform(com_ijk[0], com_ijk[1], com_ijk[2], affine_matrix)
    if abs(float(x)) <= float(midline_tolerance_mm):
        hemisphere = "none"
    else:
        hemisphere = "L" if float(x) < 0 else "R"
    return float(x), float(y), float(z), hemisphere

def _top_k_overlaps_from_integer_label_map(mask_3d, integer_label_map_3d, label_names_list, k=3, labels_to_ignore=None):
    """
    Returns a list of (label_name, coverage_fraction) sorted by coverage desc.
    - mask_3d: boolean mask of the ROI in parcellation space
    - integer_label_map_3d: integer atlas labels aligned to the same grid
    - label_names_list: list where index corresponds to integer label
    - labels_to_ignore: set of label strings to exclude (optional)
    """
    if labels_to_ignore is None:
        labels_to_ignore = set()

    label_ids, label_counts = np.unique(integer_label_map_3d[mask_3d], return_counts=True)

    rows = []
    for label_id, count_value in zip(label_ids, label_counts):
        label_id_int = int(label_id)
        if label_id_int == 0:
            continue
        label_name = label_names_list[label_id_int] if label_id_int < len(label_names_list) else f"Label{label_id_int}"
        if label_name in labels_to_ignore:
            continue
        rows.append((label_name, int(count_value)))

    if len(rows) == 0:
        return [("Unlabeled", 0.0)] * int(k)

    total_count = float(sum(count_value for _, count_value in rows))
    rows_sorted = sorted(
        [(label_name, count_value / total_count) for label_name, count_value in rows],
        key=lambda item: item[1],
        reverse=True)

    rows_sorted = rows_sorted[:int(k)]
    if len(rows_sorted) < int(k):
        rows_sorted = rows_sorted + [("Unlabeled", 0.0)] * (int(k) - len(rows_sorted))

    return rows_sorted

def _top_k_overlaps_from_yeo_map(mask_3d, yeo_integer_map_3d, yeo_name_dict, k=3):
    """
    Returns a list of (network_name, coverage_fraction) sorted by coverage desc.
    """
    label_ids, label_counts = np.unique(yeo_integer_map_3d[mask_3d], return_counts=True)

    rows = []
    for label_id, count_value in zip(label_ids, label_counts):
        label_id_int = int(label_id)
        if label_id_int == 0:
            continue
        network_name = yeo_name_dict.get(label_id_int, f"Net{label_id_int}")
        rows.append((network_name, int(count_value)))

    if len(rows) == 0:
        return [("", 0.0)] * int(k)

    total_count = float(sum(count_value for _, count_value in rows))
    rows_sorted = sorted(
        [(network_name, count_value / total_count) for network_name, count_value in rows],
        key=lambda item: item[1],
        reverse=True)

    rows_sorted = rows_sorted[:int(k)]
    if len(rows_sorted) < int(k):
        rows_sorted = rows_sorted + [("", 0.0)] * (int(k) - len(rows_sorted))

    return rows_sorted

# Determine ROI label list from the loaded parcellation data (already computed in the atlas-load cell)
roi_labels_in_parcellation = roi_labels_present

rows = []
for roi_label in roi_labels_in_parcellation:
    roi_mask = (parcellation_label_data_int == int(roi_label))
    if not roi_mask.any():
        # Should not happen if roi_labels_present came from np.unique, but keep explicit guard
        continue

    # Centroid + hemisphere (MNI coordinates)
    centroid_x, centroid_y, centroid_z, hemisphere = _compute_centroid_xyz_and_hemi(
        mask_3d=roi_mask,
        affine_matrix=parcellation_affine,
        midline_tolerance_mm=5.0)

    # Harvard–Oxford: combine cortical + subcortical max-prob overlaps (same as old code style)
    ho_cort_top3 = _top_k_overlaps_from_integer_label_map(
        mask_3d=roi_mask,
        integer_label_map_3d=harvard_oxford_cortical_maxprob_data_int,
        label_names_list=harvard_oxford_cortical_region_names,
        k=3,
        labels_to_ignore=GENERIC_ANATOMY_LABELS)

    ho_sub_top3 = _top_k_overlaps_from_integer_label_map(
        mask_3d=roi_mask,
        integer_label_map_3d=harvard_oxford_subcortical_maxprob_data_int,
        label_names_list=harvard_oxford_subcortical_region_names,
        k=3,
        labels_to_ignore=GENERIC_ANATOMY_LABELS)

    # Merge cortical + subcortical rows by simple concatenation of voxel counts logic:
    # We recreate the "combined table then normalize" behavior used in the old code.
    # To do this, we recompute from raw overlap counts rather than from the already-normalized top3 lists.
    ho_combined_label_ids_cort, ho_combined_counts_cort = np.unique(
        harvard_oxford_cortical_maxprob_data_int[roi_mask], return_counts=True)
    ho_combined_label_ids_sub, ho_combined_counts_sub = np.unique(
        harvard_oxford_subcortical_maxprob_data_int[roi_mask], return_counts=True)

    ho_combined_rows = []
    for label_id, count_value in zip(ho_combined_label_ids_cort, ho_combined_counts_cort):
        label_id_int = int(label_id)
        if label_id_int == 0:
            continue
        label_name = (
            harvard_oxford_cortical_region_names[label_id_int]
            if label_id_int < len(harvard_oxford_cortical_region_names)
            else f"Label{label_id_int}")
        if label_name in GENERIC_ANATOMY_LABELS:
            continue
        ho_combined_rows.append((label_name, int(count_value)))

    for label_id, count_value in zip(ho_combined_label_ids_sub, ho_combined_counts_sub):
        label_id_int = int(label_id)
        if label_id_int == 0:
            continue
        label_name = (
            harvard_oxford_subcortical_region_names[label_id_int]
            if label_id_int < len(harvard_oxford_subcortical_region_names)
            else f"Label{label_id_int}")
        if label_name in GENERIC_ANATOMY_LABELS:
            continue
        ho_combined_rows.append((label_name, int(count_value)))

    if len(ho_combined_rows) == 0:
        ho_top3_combined = [("Unlabeled", 0.0)] * 3
    else:
        ho_total = float(sum(count_value for _, count_value in ho_combined_rows))
        ho_props_sorted = sorted(
            [(label_name, count_value / ho_total) for label_name, count_value in ho_combined_rows],
            key=lambda item: item[1],
            reverse=True)
        ho_top3_combined = ho_props_sorted[:3]
        if len(ho_top3_combined) < 3:
            ho_top3_combined = ho_top3_combined + [("Unlabeled", 0.0)] * (3 - len(ho_top3_combined))

    main_anatomy_name, main_anatomy_fraction = ho_top3_combined[0]
    secondary_anatomy_name, secondary_anatomy_fraction = ho_top3_combined[1]
    tertiary_anatomy_name, tertiary_anatomy_fraction = ho_top3_combined[2]

    main_anatomy_coverage_percent = _percent(main_anatomy_fraction)
    secondary_anatomy_coverage_percent = _percent(secondary_anatomy_fraction)
    tertiary_anatomy_coverage_percent = _percent(tertiary_anatomy_fraction)

    anatomy_ambiguous = bool(main_anatomy_coverage_percent < int(PRIMARY_LOW_THRESH_PERCENT))

    # Yeo overlaps (top-3)
    yeo7_top3 = _top_k_overlaps_from_yeo_map(
        mask_3d=roi_mask,
        yeo_integer_map_3d=yeo_7_networks_data_int,
        yeo_name_dict=yeo_7_network_names,
        k=3)

    yeo17_top3 = _top_k_overlaps_from_yeo_map(
        mask_3d=roi_mask,
        yeo_integer_map_3d=yeo_17_networks_data_int,
        yeo_name_dict=yeo_17_network_names,
        k=3)

    rows.append({
        "roi_id": str(int(roi_label)),  # no need to pad here; padding is optional and can be derived later

        "ambiguous": anatomy_ambiguous,
        "hemi": hemisphere,

        "anatomical_primary": main_anatomy_name,
        "anatomical_primary_cov": main_anatomy_coverage_percent,
        "anatomical_secondary": secondary_anatomy_name,
        "anatomical_secondary_cov": secondary_anatomy_coverage_percent,
        "anatomical_tertiary": tertiary_anatomy_name,
        "anatomical_tertiary_cov": tertiary_anatomy_coverage_percent,

        "functional_coarse_1": yeo7_top3[0][0],
        "functional_coarse_1_cov": _percent(yeo7_top3[0][1]),
        "functional_coarse_2": yeo7_top3[1][0],
        "functional_coarse_2_cov": _percent(yeo7_top3[1][1]),
        "functional_coarse_3": yeo7_top3[2][0],
        "functional_coarse_3_cov": _percent(yeo7_top3[2][1]),

        "functional_fine_1": yeo17_top3[0][0],
        "functional_fine_1_cov": _percent(yeo17_top3[0][1]),
        "functional_fine_2": yeo17_top3[1][0],
        "functional_fine_2_cov": _percent(yeo17_top3[1][1]),
        "functional_fine_3": yeo17_top3[2][0],
        "functional_fine_3_cov": _percent(yeo17_top3[2][1]),

        # Keep coordinates available for downstream QC/visualization even if we later drop them for CSV
        "x": centroid_x,
        "y": centroid_y,
        "z": centroid_z})

label_table_dataframe = pd.DataFrame(rows).sort_values("roi_id", key=lambda series: series.astype(int)).reset_index(drop=True)

# Create manual_label column (if does not exist / is all empty), matching old script behavior
if ("manual_label" not in label_table_dataframe.columns or
    label_table_dataframe["manual_label"].replace("", np.nan).isna().all()):
    label_table_dataframe.insert(1, "manual_label", "")

# Reorder columns to match your preferred layout (same as old script)
label_table_dataframe = label_table_dataframe[
    [
        "roi_id",
        "manual_label",
        "ambiguous",
        "hemi",
        "anatomical_primary",
        "anatomical_primary_cov",
        "anatomical_secondary",
        "anatomical_secondary_cov",
        "anatomical_tertiary",
        "anatomical_tertiary_cov",
        "functional_coarse_1",
        "functional_coarse_1_cov",
        "functional_coarse_2",
        "functional_coarse_2_cov",
        "functional_coarse_3",
        "functional_coarse_3_cov",
        "functional_fine_1",
        "functional_fine_1_cov",
        "functional_fine_2",
        "functional_fine_2_cov",
        "functional_fine_3",
        "functional_fine_3_cov",
        # "x",
        # "y",
        # "z"
        ]]

print(f"[INFO] Built label table for atlas_tag={atlas_tag} with {label_table_dataframe.shape[0]} ROIs.")
display(label_table_dataframe.head(10))

In [ ]:
# __________________________________________________________________________________________________________
### MANUAL LABELING UTILITIES (VISUALIZE ROI + SET/LOAD MANUAL LABELS)

mni_anatomical_template = datasets.load_mni152_template()

def format_roi_id_for_table(roi_id_value):
    """
    Keep ROI IDs consistent with label_table_dataframe['roi_id'].
    In our current build step, roi_id is stored as a string of the integer label (no padding).
    """
    return str(int(roi_id_value))

def format_roi_id_for_filename(roi_id_value, width=None):
    """
    For filenames only. Width can be roi_id_string_width (computed earlier) or user-specified.
    """
    roi_id_int = int(roi_id_value)
    if width is None:
        return str(roi_id_int)
    return str(roi_id_int).zfill(int(width))

def visualize_roi_with_text(roi_id, show_coords=True, save_path=None, show=True, filename_pad_width=None):
    roi_id_string = format_roi_id_for_table(roi_id)
    roi_id_int = int(roi_id_string)

    if roi_id_string not in set(label_table_dataframe["roi_id"].tolist()):
        raise ValueError(f"ROI {roi_id_string} not found in label_table_dataframe['roi_id'].")

    roi_mask_bool = (parcellation_label_data_int == roi_id_int)
    if not roi_mask_bool.any():
        raise ValueError(f"ROI {roi_id_int} has no voxels in parcellation_label_data_int.")

    roi_mask_image = nib.Nifti1Image(
        roi_mask_bool.astype(np.int16),
        parcellation_label_image.affine,
        parcellation_label_image.header)

    if show_coords:
        roi_com_ijk = center_of_mass(roi_mask_bool)
        x, y, z = coord_transform(
            roi_com_ijk[0], roi_com_ijk[1], roi_com_ijk[2], parcellation_label_image.affine)
        cut_coords = (float(x), float(y), float(z))
    else:
        cut_coords = None

    row = label_table_dataframe.loc[label_table_dataframe["roi_id"] == roi_id_string].iloc[0]

    lines = [
        f"ROI {roi_id_string} — {row.get('manual_label','')}",
        f"Hemi: {row.get('hemi')}",
        f"Ambiguous: {row.get('ambiguous','')}",
        "",
        f"Anatomical Primary:   {row.get('anatomical_primary','')} ({row.get('anatomical_primary_cov','')}%)",
        f"Anatomical Secondary: {row.get('anatomical_secondary','')} ({row.get('anatomical_secondary_cov','')}%)",
        f"Anatomical Tertiary:  {row.get('anatomical_tertiary','')} ({row.get('anatomical_tertiary_cov','')}%)",
        "",
        f"Functional Coarse 1: {row.get('functional_coarse_1','')} ({row.get('functional_coarse_1_cov','')}%)",
        f"Functional Coarse 2: {row.get('functional_coarse_2','')} ({row.get('functional_coarse_2_cov','')}%)",
        f"Functional Coarse 3: {row.get('functional_coarse_3','')} ({row.get('functional_coarse_3_cov','')}%)",
        "",
        f"Functional Fine 1: {row.get('functional_fine_1','')} ({row.get('functional_fine_1_cov','')}%)",
        f"Functional Fine 2: {row.get('functional_fine_2','')} ({row.get('functional_fine_2_cov','')}%)",
        f"Functional Fine 3: {row.get('functional_fine_3','')} ({row.get('functional_fine_3_cov','')}%)"]
    wrapped_text = "\n".join(textwrap.fill(line, width=90) for line in lines)

    figure = plt.figure(figsize=(8, 10))
    grid_spec = figure.add_gridspec(2, 1, height_ratios=[3, 1])

    axis_brain = figure.add_subplot(grid_spec[0])
    display_object = plotting.plot_roi(
        roi_img=roi_mask_image,
        bg_img=mni_anatomical_template,
        display_mode="ortho",
        cut_coords=cut_coords,
        draw_cross=True,
        title=f"{atlas_tag} ROI {format_roi_id_for_filename(roi_id_int, filename_pad_width)}",
        cmap="autumn",
        axes=axis_brain)

    try:
        display_object.annotate(False)
    except Exception:
        pass

    axis_text = figure.add_subplot(grid_spec[1])
    axis_text.axis("off")
    axis_text.text(0, 1, wrapped_text, fontsize=10, va="top", family="monospace")

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(save_path, dpi=150)

    if show:
        plt.show()
    else:
        plt.close(figure)

    return None

def set_manual_label(roi_id, label_text):
    """
    Set manual label for a given ROI.
    Guardrail:
    - If label_text is empty or whitespace-only, do nothing.
    """
    if label_text is None or str(label_text).strip() == "":
        # Explicit no-op to avoid accidental overwrites
        print("[INFO] Empty manual label provided; no changes made.")
        return
    roi_id_string = format_roi_id_for_table(roi_id)
    mask = label_table_dataframe["roi_id"] == roi_id_string
    if not mask.any():
        raise ValueError(f"ROI {roi_id_string} not found in label_table_dataframe['roi_id'].")
    label_table_dataframe.loc[mask, "manual_label"] = str(label_text)

def load_existing_manual_labels(label_csv_path):
    """
    If an existing labels CSV is present, merge its manual_label column into the current table.

    Rules:
    - Uses roi_id as the join key (stringified integer labels).
    - Only overwrites current manual_label when the existing manual_label is non-empty.
    - Leaves all other columns untouched.
    """
    label_csv_path = Path(label_csv_path)
    if not label_csv_path.exists():
        print(f"[INFO] No existing manual ROI labels detected (at: {label_csv_path}).")
        return

    existing_labels_dataframe = pd.read_csv(label_csv_path, dtype={"roi_id": str})
    if "roi_id" not in existing_labels_dataframe.columns:
        raise ValueError(f"Existing label CSV missing required column 'roi_id': {label_csv_path}")
    if "manual_label" not in existing_labels_dataframe.columns:
        print(f"[INFO] Existing label CSV has no 'manual_label' column: {label_csv_path} (nothing to import).")
        return

    existing_labels_dataframe["roi_id"] = existing_labels_dataframe["roi_id"].astype(str).str.strip()
    existing_labels_dataframe["manual_label"] = existing_labels_dataframe["manual_label"].fillna("").astype(str)

    # Build a lookup from roi_id -> manual_label (non-empty only)
    existing_manual_label_lookup = {
        row_roi_id: row_manual_label
        for row_roi_id, row_manual_label in zip(
            existing_labels_dataframe["roi_id"].tolist(),
            existing_labels_dataframe["manual_label"].tolist())
        if str(row_manual_label).strip() != ""}

    if len(existing_manual_label_lookup) == 0:
        print(f"[INFO] Existing label CSV contains no non-empty manual labels: {label_csv_path}")
        return

    # Apply lookup to current table without changing empty current labels unless overwrite is meaningful
    updated_count = 0
    for index_value in range(label_table_dataframe.shape[0]):
        current_roi_id = str(label_table_dataframe.loc[index_value, "roi_id"]).strip()
        if current_roi_id in existing_manual_label_lookup:
            label_table_dataframe.loc[index_value, "manual_label"] = existing_manual_label_lookup[current_roi_id]
            updated_count += 1

    print(f"[INFO] Importing {updated_count} manual labels from: {label_csv_path}")

existing_label_csv_path = LABEL_OUTPUT_PATH / f"ROI_labels__{atlas_tag}.csv"
load_existing_manual_labels(existing_label_csv_path)


--------
--------
--------
--------
--------
# Manual label-editing section:

In [ ]:
roi_to_review = 1

visualize_roi_with_text(
    roi_to_review,
    show_coords=True,
    save_path=None,
    show=True,
    filename_pad_width=roi_id_string_width)

In [ ]:
set_manual_label(roi_to_review, "")

label_table_dataframe.loc[
    label_table_dataframe["roi_id"] == str(int(roi_to_review)),
    ["roi_id", "manual_label"]]

In [ ]:
label_table_dataframe[['roi_id', 'manual_label']]

-------
-------
-------
-------
-------

In [ ]:
# __________________________________________________________________________________________________________
### EXPORT MAIN LABEL TABLE (ALWAYS) + OPTIONAL FULL ROI DETAIL PLOTS (PNG)

# -------------------------
# Export main table:
# -------------------------
label_table_output_csv_path = LABEL_OUTPUT_PATH / f"LABELS_{atlas_tag}.csv"

# Save roi_id as a stringified integer (no padding) to preserve the "native" ROI labels
label_table_dataframe_to_save = label_table_dataframe.copy()
label_table_dataframe_to_save["roi_id"] = label_table_dataframe_to_save["roi_id"].astype(str).str.strip()

label_table_dataframe_to_save.to_csv(label_table_output_csv_path, index=False)
print(f"[INFO] Saved label table CSV:\n    {label_table_output_csv_path}\n")

# -------------------------
# Export per-ROI detail plots (as PNGs)
# -------------------------
if bool(EXPORT_FULL_ROI_REPORTS):
    print(f"[INFO] EXPORT_FULL_ROI_REPORTS=True; exporting ROI plots to:\n    {LABEL_OUTPUT_PATH}\n")

    roi_count = 0
    for roi_label in roi_labels_present:
        roi_label_int = int(roi_label)

        # Filename must be ROI-<original_INT_label>.png in the main output directory
        roi_id_for_filename = format_roi_id_for_filename(roi_label_int, width=roi_id_string_width)
        roi_plot_output_png_path = LABEL_OUTPUT_PATH / f"ROI-{roi_id_for_filename}.png"

        visualize_roi_with_text(
            roi_label_int,
            show_coords=True,
            save_path=str(roi_plot_output_png_path),
            show=False,
            filename_pad_width=roi_id_string_width)
        roi_count += 1
    print(f"\n[INFO] Finished exporting ROI plots: wrote {roi_count} PNG files.")
else:
    print("\n[INFO] EXPORT_FULL_ROI_REPORTS=False; skipping ROI plot export.")

-------